# Training YOLO26n-Depth di laptop (Windows + NVIDIA GPU)

Notebook ini menjalankan `train_arsitektur.py` dari folder proyek—tidak memakai Google Colab atau Google Drive. Pastikan kernel notebook memakai virtual environment proyek dan PyTorch yang mendukung CUDA. Dataset dan checkpoint disimpan lokal di `datasets/` dan `runs/arsitektur-laptop/`.

RTX 4050 laptop umumnya memiliki VRAM terbatas; konfigurasi awal memakai batch 2, mixed precision (AMP), dan 480×640. Kalau GPU kehabisan memori, turunkan `BATCH_SIZE` menjadi 1. Untuk melanjutkan training, notebook memakai `last.pt`; naikkan `EPOCHS` bila jumlah epoch yang diminta sudah tercapai.

In [1]:
from pathlib import Path
import sys
import torch

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'train_arsitektur.py').is_file() and (path / 'arsitektur.py').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari folder proyek yang berisi train_arsitektur.py.')

print('Project:', ROOT)
print('Python:', sys.executable)
print('PyTorch:', torch.__version__)
print('CUDA tersedia:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('PyTorch di kernel ini tidak melihat CUDA. Pasang build PyTorch CUDA dari https://pytorch.org/get-started/locally/, lalu restart kernel.')

GPU = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {GPU} ({VRAM_GB:.1f} GiB VRAM)')

Project: c:\Users\ACER\Downloads\Comvis\code
Python: c:\Users\ACER\AppData\Local\Programs\Python\Python310\python.exe
PyTorch: 2.5.1+cu121
CUDA tersedia: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU (6.0 GiB VRAM)


## Konfigurasi

Training menggabungkan SUN RGB-D untuk deteksi + depth dengan COCO untuk deteksi saja (COCO tidak punya label depth). `COCO_RATIO = 0.3` menargetkan sekitar 30% data training dari COCO; trainer mengunduh anotasi COCO dan hanya gambar dengan kelas target. Untuk menghindari mismatch sertifikat di `images.cocodataset.org`, trainer mengambil file dari endpoint HTTPS path-style Amazon S3 untuk bucket COCO yang sama—verifikasi sertifikat tetap aktif. Path SUN RGB-D default mengikuti `datasets/SUNRGBD/`, anotasi COCO disimpan di `datasets/coco/annotations/`, dan gambar terpilih di `datasets/coco/train2017/`. Siapkan internet dan ruang disk tambahan.

In [2]:
DATASET_ROOT = ROOT / 'datasets'
SUN_ROOT = DATASET_ROOT / 'SUNRGBD'
RUN_DIR = ROOT / 'runs' / 'arsitektur-laptop-v2'
COCO_ANNOTATIONS = DATASET_ROOT / 'coco' / 'annotations' / 'instances_train2017.json'

CLASSES = ('person', 'laptop', 'cell phone', 'bottle')
IMAGE_SIZE = (480, 640)  # tinggi, lebar; harus kelipatan 32
DEPTH_MAX_M = 10.0
EPOCHS = 50
BATCH_SIZE = 2
NUM_WORKERS = 0  # aman untuk Windows; bisa dicoba 2 jika loading data lambat
COCO_RATIO = 0.3
SEED = 42
SMOKE_TEST = False  # True untuk tes singkat pada maksimal 200 train / 40 val sampel
RESUME_FROM = RUN_DIR / 'last.pt'  # otomatis lanjut jika checkpoint ini ada

print({
    'sun_root': str(SUN_ROOT),
    'output_dir': str(RUN_DIR),
    'classes': CLASSES,
    'epochs': 1 if SMOKE_TEST else EPOCHS,
    'batch_size': BATCH_SIZE,
    'image_size': IMAGE_SIZE,
    'coco_ratio': COCO_RATIO,
    'coco_annotations': str(COCO_ANNOTATIONS) if COCO_ANNOTATIONS.is_file() else 'unduh otomatis jika tidak diisi',
    'resume_from': str(RESUME_FROM) if RESUME_FROM.is_file() else 'mulai dari awal',
})

{'sun_root': 'c:\\Users\\ACER\\Downloads\\Comvis\\code\\datasets\\SUNRGBD', 'output_dir': 'c:\\Users\\ACER\\Downloads\\Comvis\\code\\runs\\arsitektur-laptop-v2', 'classes': ('person', 'laptop', 'cell phone', 'bottle'), 'epochs': 50, 'batch_size': 2, 'image_size': (480, 640), 'coco_ratio': 0.3, 'coco_annotations': 'c:\\Users\\ACER\\Downloads\\Comvis\\code\\datasets\\coco\\annotations\\instances_train2017.json', 'resume_from': 'mulai dari awal'}


## Jalankan training

Setiap epoch menyimpan `last.pt`; `best.pt` menyimpan checkpoint dengan validation loss terbaik. Jika proses terhenti, jalankan ulang notebook untuk melanjutkan dari `last.pt`.

In [3]:
import subprocess

command = [
    sys.executable, '-u', str(ROOT / 'train_arsitektur.py'),
    '--sun-root', str(SUN_ROOT),
    '--download-dir', str(DATASET_ROOT),
    '--output-dir', str(RUN_DIR),
    '--classes', *CLASSES,
    '--img-size', str(IMAGE_SIZE[0]), str(IMAGE_SIZE[1]),
    '--depth-max-m', str(DEPTH_MAX_M),
    '--epochs', str(1 if SMOKE_TEST else EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--workers', str(NUM_WORKERS),
    '--coco-ratio', str(COCO_RATIO),
    '--seed', str(SEED),
    '--device', 'cuda',
    '--amp',
]
if COCO_ANNOTATIONS.is_file():
    command += ['--coco-annotations', str(COCO_ANNOTATIONS)]
if SMOKE_TEST:
    command += ['--max-train-samples', '200', '--max-val-samples', '40']
if RESUME_FROM.is_file():
    command += ['--resume', str(RESUME_FROM)]

print('Menjalankan training di GPU:', GPU, flush=True)
process = subprocess.Popen(
    command,
    cwd=ROOT,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    encoding='utf-8',
    errors='replace',
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, command)

Menjalankan training di GPU: NVIDIA GeForce RTX 4050 Laptop GPU
SUN RGB-D depth rotation: on; object index base: 0
Skipping SUN RGB-D scene with invalid annotation: Could not read SUN RGB-D annotation C:\Users\ACER\Downloads\Comvis\code\datasets\SUNRGBD\SUNRGBD\kv1\b3dodata\img_0139\annotation2Dfinal\index.json
Skipping SUN RGB-D scene with invalid annotation: Could not read SUN RGB-D annotation C:\Users\ACER\Downloads\Comvis\code\datasets\SUNRGBD\SUNRGBD\kv1\b3dodata\img_0174\annotation2Dfinal\index.json
Found 2001 SUN RGB-D records; depth rotation enabled.
Selected 749 COCO images for classes ('person', 'laptop', 'cell phone', 'bottle').
All 749 selected COCO images are already available.
Using 749 COCO images (target ratio 30%).
Train/val records: 2497 / 253 | boxes: {'person': 2046, 'laptop': 445, 'cell phone': 317, 'bottle': 1441}

[|] Epoch 1/50: loading train batch 1/1249
[/] Epoch 1/50: training batch 1/1249
[-] Epoch 1/50: training batch 1/1249
[\] Epoch 1/50: training batch 1

KeyboardInterrupt: 

In [ ]:
BEST_CHECKPOINT = RUN_DIR / 'best.pt'
if not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Checkpoint terbaik belum ada: {BEST_CHECKPOINT}')
checkpoint = torch.load(BEST_CHECKPOINT, map_location='cpu', weights_only=True)
print('Checkpoint terbaik:', BEST_CHECKPOINT)
print('Epoch:', checkpoint['epoch'])
print('Validation loss terbaik:', checkpoint['best_val_loss'])
print('Untuk aplikasi kamera: salin checkpoint ini ke root proyek atau jalankan main.py --model dengan path checkpoint tersebut.')